In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [ ]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams
from utils import MusicDBPermDir
from timeutils import Timestat
from sys import prefix
from pandas import date_range, concat, Series, DataFrame
from datetime import datetime
mp = MasterParams(verbose=False)
io = FileIO()
mdbpd = MusicDBPermDir()

# Global Variables

In [ ]:
countries = ['sa', 'ie', 'ar', 'no', 'ma', 'in', 'cl', 'hu', 'ch', 'ru', 'lu', 'pt', 'bg', 'jp', 'hn', 'se', 'vn', 'ro', 'hk', 'ec', 'cr', 'nz', 
             'id', 'gb', 'fr', 'sk', 'pe', 'pa', 'lt', 'br', 'ua', 'gr', 'it', 'es', 'fi', 'eg', 'cz', 'uy', 'il', 'py', 'do', 'is', 'at', 'dk',
             'sg', 'ni', 'ca', 'be', 'mx', 'tr', 'de', 'co', 'tw', 'us', 'ae', 'gt', 'pl', 'sv', 'th', 'nl', 'ee', 'ph', 'bo', 'au', 'my', 'za', 'lv']
categoryDir = DirInfo("/Volumes/Piggy/Charts/data/spotify/categories")
resultsDir  = DirInfo("/Volumes/Piggy/Charts/data/spotify/results")

# Manual Downloads

In [ ]:
start = '11/1/2020'
end   = datetime.now().strftime("%Y-%m-%d")
datevals = [x.strftime('%Y-%m-%d') for x in list(date_range(start=start, end=end, freq='W-FRI'))]
datevals.reverse()

In [ ]:
toget = {}
for country in countries:    
    nDownload = 0
    errs = 0
    N = len(datevals)
    for i,dateval in enumerate(datevals):
        if errs > 1:
            break
        if nDownload > 3:
            break
        if i == len(datevals)-1:
            break
        start = datevals[i+1]
        end   = datevals[i]
        url      = "https://spotifycharts.com/regional/{0}/weekly/{1}--{2}/download".format(country, start, end)
        #savename = "/Users/tgadfort/Documents/code/charts/spotify/
        
        basename = "regional-{0}-weekly-{1}--{2}".format(country, start, end)
        savename = categoryDir.join("{0}.p".format(basename))
        if savename.exists():
            continue
        savename = categoryDir.join("{0}.csv".format(basename))
        if savename.exists():
            continue
        savename = FileInfo("/Users/tgadfort/Downloads/{0}.csv".format(basename))
        if savename.exists():
            continue
            
        if toget.get(country) is None:
            toget[country] = {}
        toget[country][dateval] = url
        
numToGet = Series(toget).apply(lambda x: len(x))
numToGet.name = "NumToGet"
urlsToGet = DataFrame(numToGet).join(Series(toget, name="URLs")).sort_values(by="NumToGet", ascending=True)

In [ ]:
for i,(idx,row) in enumerate(urlsToGet.iterrows()):
    print("{0} [{1}/{2}] -- {3} -- {4}".format("="*10,i,len(urlsToGet),idx,"="*10))
    urls = Series(row["URLs"])
    
    urlsToDownload = urls.sample(n=5) if len(urls) >= 5 else urls
    for j,(dateval,url) in enumerate(urlsToDownload.iteritems()):
        if j % 1 == 0:
            print("  ",j,'/',len(urls),'\t',dateval,'\t',url)

# Parsing

In [ ]:
from glob import glob
#from dbUtils import utilsSpotifyCharts
#util = utilsSpotifyCharts()

def getArtistID(artistName):
    return util.getArtistID(artistName)


def getSpotifyData(ifile):
    if FileInfo(ifile).ext == ".p":
        try:
            csvData = read_csv(StringIO(getHTML(ifile).find("p").text), skiprows=1)
        except:
            csvData = None
        return csvData
    elif FileInfo(ifile).ext == ".csv":
        try:
            csvData = read_csv(ifile, skiprows=1)
        except:
            csvData = None
        return csvData
    
    
def getSpotifyCountryFiles(country):
    pfiles = list(categoryDir.glob("regional-{0}-weekly*.p".format(country)))
    cfiles = list(categoryDir.glob("regional-{0}-weekly*.csv".format(country)))
    #pfiles  = glob("/Volumes/Piggy/Charts/data/spotify/categories/regional-{0}-weekly*.p".format(country))
    #cfiles  = glob("/Volumes/Piggy/Charts/data/spotify/categories/regional-{0}-weekly*.csv".format(country))
    return pfiles,cfiles

    
def getSpotifyCountryData(country):
    print("Getting Spotify Country [{0}] Data".format(country))
    pfiles,cfiles = getSpotifyCountryFiles(country)
    print("  Found {0} and {1} .p/.csv files".format(len(pfiles),len(cfiles)))
    spotifyData = concat([getSpotifyData(ifile) for ifile in pfiles+cfiles])
    print("  Found {0} entries".format(spotifyData.shape[0]))    
    return spotifyData


def groupSpotifyCountryData(country, force=False):
    savename = resultsDir.join("regional-{0}-weekly.p".format(country))
    #savename = "/Volumes/Piggy/Charts/data/spotify/results/regional-{0}-weekly.p".format(country)
    if FileInfo(savename).exists:
        if force is False:
            return
    
    spotifyData = getSpotifyCountryData(country)
    artistSpotifyCountryData = {artistName: {trackName: trackData['Streams'].sum() for trackName,trackData in artistData.groupby("Track Name")}
                                  for artistName,artistData in spotifyData.groupby("Artist")}
              
    print("   ==> {0: <8}{1: <8}{2}".format(country,len(artistSpotifyCountryData),sum([len(tracks) for tracks in artistSpotifyCountryData.values()])))
    io.save(idata=artistSpotifyCountryData, ifile=savename)
    print("   ==> Saved Results To {0}".format(savename))

In [ ]:
from zipfile import ZipFile
#from StringIO import StringIO
def extract_zip(input_zip):
    input_zip=ZipFile(input_zip)
    return {name: input_zip.read(name) for name in input_zip.namelist()}

In [ ]:
ts = Timestat("Extracting ZIP")
retval = extract_zip("/Users/tgadfort/Downloads/archive (1).zip")
ts.stop()

In [ ]:
for k,v in retval.items():
    print(k)
    print(type(v))

In [ ]:
N = len(countries)
ts = Timestat("Loading Spotify Data From {0} Countries".format(N))
for n,country in enumerate(countries):
    if (n+1) % 5 == 0:
        ts.update(n=n+1,N=N)
    groupSpotifyCountryData(country, force=True)
ts.stop()

# Combine External Downloads

In [ ]:
ts = timestat("Getting Combined CSV Data")
fullData = io.get("/Volumes/Piggy/Charts/data/spotify/full/charts.csv")
ts.stop()

In [ ]:
fullData.head()

In [ ]:
ts = timestat("Grouping Full Data")
artistSpotifyFullData = {artistName: {trackName: trackData['streams'].sum() for trackName,trackData in artistData.groupby("title")}
                         for artistName,artistData in fullData.groupby("artist")}
ts.stop()

In [ ]:
io.save(idata=artistSpotifyFullData, ifile="/Volumes/Piggy/Charts/data/spotify/full/artistSpotifyData.p")

# Combine Country Data

In [ ]:
artistSpotifyData = {}
for country in countries:
    savename = "/Volumes/Piggy/Charts/data/spotify/results/regional-{0}-weekly.p".format(country)
    if fileUtil(savename).exists:
        artistSpotifyData[country] = io.get(savename)    
#io.save(idata=artistSpotifyData, ifile="/Volumes/Piggy/Charts/data/spotify/results/artistSpotifyData.p")

In [ ]:
saveCountryData = False

artistSummaryData = {}
for country,countryData in artistSpotifyData.items():
    for artistName,artistNameData in countryData.items():
        if artistSummaryData.get(artistName) is None:
            artistSummaryData[artistName] = {}
        for trackName,trackStreams in artistNameData.items():
            if saveCountryData:
                if artistSummaryData[artistName].get(trackName) is None:
                    artistSummaryData[artistName][trackName] = {}
                if artistSummaryData[artistName][trackName].get(country) is None:
                    artistSummaryData[artistName][trackName][country] = 0
                artistSummaryData[artistName][trackName][country] += trackStreams
            else:
                if artistSummaryData[artistName].get(trackName) is None:
                    artistSummaryData[artistName][trackName] = 0
                artistSummaryData[artistName][trackName] += trackStreams
                
    print(country,'\t',len(artistSummaryData),'\t',sum([len(tracks) for tracks in artistSummaryData.values()]))
artistSummaryData = Series(artistSummaryData)

In [ ]:
artistStreams = artistSummaryData.apply(lambda x: Series(x).sum())
artistStreams.name = "Streams"

In [ ]:
artistSpotifyDF = DataFrame(artistStreams)
artistSpotifyDF.index.name = "ArtistName"

tracks = artistSummaryData.apply(lambda x: list(x.keys()))
tracks.name = "Tracks"
artistSpotifyDF = artistSpotifyDF.join(tracks)
artistSpotifyDF["NumTracks"] = artistSpotifyDF["Tracks"].apply(len)

artistSpotifyDF = artistSpotifyDF.reset_index().sort_values(by="Streams", ascending=False)

In [ ]:
print("Total Artists: {0}".format(artistSpotifyDF.shape[0]))
artistSpotifyDF.head()

In [ ]:
io.save(idata=artistSpotifyDF, ifile="/Volumes/Piggy/Charts/data/spotify/results/SpotifyArtistData.p")

# Track Renames

In [ ]:
io = fileIO()
artistSpotifyDF = io.get("/Volumes/Piggy/Charts/data/spotify/results/SpotifyArtistData.p")
artistSpotifyDF.index = artistSpotifyDF["ArtistName"].apply(getArtistID)
artistSpotifyDF.index.name = "ArtistID"

In [ ]:
#tracksDF = [{"TrackName": trackName, "ArtistName": row["ArtistName"]} for trackName in row["Tracks"]} 
artistTracks = []
for idx,row in artistSpotifyDF.iterrows():
    artistTracks += [{"TrackID": getArtistID(" ".join([trackName,row["ArtistName"]])), "ArtistName": row["ArtistName"], "TrackName": trackName} for trackName in row["Tracks"]]

In [ ]:
tracksDF = Series(artistTracks).apply(Series)

In [ ]:
io.save(idata=tracksDF, ifile="/Volumes/Piggy/Charts/data/spotify/results/SpotifyTracksData.p")

# Parse Tracks

In [ ]:
tracksDF = io.get("/Volumes/Piggy/Charts/data/spotify/results/SpotifyTracksData.p")

In [ ]:
import regex
def getParenValues(artistName):
    return regex.findall(r'\((.*?)\)+', artistName)
def getFeatureArtist(artistName):
    return regex.findall(r'\b(feat.\s|Feat.\s|with\s)\b', artistName)

tracksDF["ParenValues"] = tracksDF["TrackName"].apply(getParenValues)
toAnalyze = tracksDF[tracksDF["ParenValues"].apply(len) > 0]
toAnalyze.head()

In [ ]:
rParen = r'\((.*?)\)+'
rFeat  = r'\b(feat.\s|Feat.\s|with\s)\b'
rSuffix= r'\s-\sRemix'

In [ ]:
from multiArtist import multiartist
ma = multiartist()

def findFeatureArtist(trackName):
    parenValue = regex.search(rParen, trackName)
    if parenValue is not None:
        featureValue = regex.search(rFeat, parenValue.group())
        if featureValue is not None:
            ## Split Track Since We Found A Featured Artist
            trackName = regex.sub(rParen, "", trackName)
            featured  = regex.sub(rFeat, "", parenValue.group())
            return {"Title": trackName, "Featured": featured[1:-1]}
    return {"Title": trackName, "Featured": None}

def findSuffix(trackName):
    suffixValue = regex.search(rSuffix, trackName)
    if suffixValue is not None:
        ## Split Track Since We Found A Suffix Artist
        trackName = regex.sub(rSuffix, "", trackName)
        return {"Title": trackName, "Suffix": suffixValue.group()}
    return {"Title": trackName, "Suffix": None}

def splitTitle(trackName):
    return trackName.split(" - ")[0].strip()

def getFeaturedArtists(featured):
    retval = list(ma.getArtistNames(featured).keys()) if featured is not None else []
    return retval

In [ ]:
tracksDF["Track"] = tracksDF['TrackName'].apply(findFeatureArtist)

In [ ]:
tracksDF["TrackTitle"]      = tracksDF["Track"].apply(lambda x: x["Title"])
tracksDF["TrackFeatured"]   = tracksDF["Track"].apply(lambda x: x["Featured"])
tracksDF["TrackSplitTitle"] = tracksDF["TrackTitle"].apply(splitTitle)
tracksDF["TrackFeaturedArtists"] = tracksDF["TrackFeatured"].apply(getFeaturedArtists)

In [ ]:
tracksCleanDF = tracksDF[["TrackID", "ArtistName", "TrackSplitTitle", "TrackFeaturedArtists"]]

In [ ]:
artistTracksData = Series({artistName: dict(zip(df["TrackID"], df["TrackSplitTitle"])) for artistName,df in tracksCleanDF.groupby("ArtistName")}, name="Tracks")

finalSpotifyDF = DataFrame(artistTracksData)
finalSpotifyDF.index.name = "ArtistName"
finalSpotifyDF = finalSpotifyDF.reset_index()

from pandas import merge
artistSpotifyDF = io.get("/Volumes/Piggy/Charts/data/spotify/results/SpotifyArtistData.p")
finalSpotifyDF  = merge(finalSpotifyDF, artistSpotifyDF[["ArtistName", "Streams"]], on="ArtistName")

In [ ]:
finalSpotifyDF.index = finalSpotifyDF["ArtistName"].apply(getArtistID)
finalSpotifyDF.index.name = "ArtistID"
finalSpotifyDF

# Create Artist Files

In [ ]:
from artistDBBase import artistDBBase, artistDBDataClass
from artistDBBase import artistDBNameClass, artistDBMetaClass, artistDBIDClass, artistDBURLClass, artistDBPageClass
from artistDBBase import artistDBProfileClass, artistDBMediaClass, artistDBMediaAlbumClass
from artistDBBase import artistDBMediaDataClass, artistDBMediaCountsClass, artistDBFileInfoClass
from artistDBBase import artistDBTextClass, artistDBLinkClass
from strUtils import fixName
from dbUtils import utilsDiscogs
from hashlib import md5

from fsUtils import setDir

def getMediaCounts(media):
    amcc = artistDBMediaCountsClass()

    credittype = "Releases"
    if amcc.counts.get(credittype) == None:
        amcc.counts[credittype] = {}
    for creditsubtype in media.media.keys():
        amcc.counts[credittype][creditsubtype] = int(len(media.media[creditsubtype]))

    return amcc

basedir = "./"
savedir = setDir(basedir, "db")
tsAll   = timestat("Creating DB Data")
Nmod    = 100

modValData = {}
N = finalSpotifyDF.shape[0]
for i,(artistID,artistIDData) in enumerate(finalSpotifyDF.iterrows()):
    artistTracks = artistIDData["Tracks"]
    artistName   = artistIDData["ArtistName"]
    artistURL    = None
    generalData  = None

 
    mediaData = {}
    mediaName = "Songs"
    mediaData[mediaName] = []
    for code, trackName in artistTracks.items():
        album        = trackName
        albumURL     = None
        albumArtists = [artistName]

        amdc = artistDBMediaDataClass(album=album, url=albumURL, artist=albumArtists, code=code, year=None)
        mediaData[mediaName].append(amdc)
        
        
    artist      = artistDBNameClass(name=artistName, err=None)
    meta        = artistDBMetaClass(title=None, url=artistURL)
    url         = artistDBURLClass(url=artistURL)
    ID          = artistDBIDClass(ID=artistID)
    pages       = artistDBPageClass(ppp=1, tot=1, redo=False, more=False)
    profile     = artistDBProfileClass(general=generalData)
    media       = artistDBMediaClass()
    media.media = mediaData
    mediaCounts = getMediaCounts(media)
    info        = artistDBFileInfoClass(info=None)

    
    modVal = int(artistID) % 100
    if modValData.get(modVal) is None:
        modValData[modVal] = {}
    modValData[modVal][artistID] = artistDBDataClass(artist=artist, meta=meta, url=url, ID=ID, pages=pages, 
                                                     profile=profile, mediaCounts=mediaCounts, media=media, info=info)
    if (i+1) %= 2500:
        tsAll.update(n=i+1, N=N)
    

for modVal,modData in modValData.items():
    outdir = setDir(basedir, "db")
    io.save(idata=Series(modData), ifile=setFile(outdir, "{0}-{1}.p".format(modVal, "DB")))
    
tsAll.stop()

In [ ]:
from masterDBGate import masterDBGate
from fsUtils import fileUtil, dirUtil, setDir, setFile
mdbGate = masterDBGate()
disc = mdbGate.getDBDisc("SpotifyCharts")
basedir = "./"
for modVal in range(100):    
    print("Saving ModVal={0}".format(modVal))
    outdir = setDir(basedir, "db")
    modValData = io.get(setFile(outdir, "{0}-{1}.p".format(modVal, "DB")))
    disc.saveDBModValData(idata=modValData, modVal=modVal)

# Spotify API

In [ ]:
!pip install spotipy

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(client_id="61e441c3b90c4873aa0e6b9582564f95",
                                                           client_secret="ae0d0f968bf443fdac1d9ac6ef65fc0f"))

results = sp.search(q='weezer', limit=20)
for idx, track in enumerate(results['tracks']['items']):
    print(idx, track['name'])

In [ ]:
# shows artist info for a URN or URL

from spotipy.oauth2 import SpotifyClientCredentials
import spotipy
import sys
import pprint

auth_manager=SpotifyClientCredentials(client_id="61e441c3b90c4873aa0e6b9582564f95", client_secret="ae0d0f968bf443fdac1d9ac6ef65fc0f")
sp = spotipy.Spotify(auth_manager=auth_manager)

In [ ]:
result.keys()

In [ ]:
urn = 'spotify:artist:3jOstUTkEu2JkjvRdBA5Gu'

#sp = spotipy.Spotify(client_credentials_manager=SpotifyClientCredentials())

artist = sp.artist(urn)
artist

In [ ]:
result = sp.search("Nirvana", limit=50, type='artist')

In [ ]:
result['artists'].keys()

In [ ]:
help(sp.search)

In [ ]:
search_str = 'Radiohead'
result = sp.search(search_str)
pprint.pprint(result)